# Funções no Pandas

Aula prática usando como base o exemplo do seu arquivo `funcao_pandas.ipynb`.

Funções deixam seu código pandas mais limpo, reutilizável e fácil de testar. Vamos ver 7 padrões, do básico ao avançado, sempre partindo do mesmo DataFrame de funcionários.

## Preparando o DataFrame

Criamos uma única vez o DataFrame que será usado em todos os exemplos abaixo.

In [32]:
import pandas as pd

dados = {
    'Nome': ['Ana', 'Bruno', 'Carlos'],
    'Idade': [25, 30, 28],
    'Salário': [3000, 4500, 3800]
}

df = pd.DataFrame(dados)
display(df)

,Nome,Idade,Salário
0,Ana,25,3000
1,Bruno,30,4500
2,Carlos,28,3800


## 1. Função recebendo um DataFrame

Este é o exemplo que você já tem. Uma função comum do Python que recebe um DataFrame como argumento — útil para encapsular cálculos repetitivos. Em vez de digitar `df['Salário'].mean()` toda hora, basta chamar `salario_medio(df)`.

In [33]:
def salario_medio(dataframe):
    return dataframe['Salário'].mean()

print(f"Salário médio: R$ {salario_medio(df):.2f}")

Salário médio: R$ 3766.67


## 2. Função com mais de um parâmetro

Adicione parâmetros para deixar a função flexível. Aqui filtramos o DataFrame por uma idade mínima qualquer — assim você reaproveita a mesma função para vários filtros.

In [34]:
def filtrar_por_idade(dataframe, idade_minima):
    return dataframe[dataframe['Idade'] >= idade_minima]

print(filtrar_por_idade(df, 28))

     Nome  Idade  Salário
1   Bruno     30     4500
2  Carlos     28     3800


## 3. `.apply()` em uma coluna

`.apply()` executa sua função para **cada valor** da coluna, retornando uma nova Series. É a forma idiomática de transformar dados elemento a elemento sem precisar escrever um `for`.

In [35]:
def aumentar_10_prcnt(valor):
    return valor * 1.10

df['Salário com Aumento'] = df['Salário'].apply(aumentar_10_prcnt)
print(df)

     Nome  Idade  Salário  Salário com Aumento
0     Ana     25     3000               3300.0
1   Bruno     30     4500               4950.0
2  Carlos     28     3800               4180.0


## 4. `.apply()` por linha (`axis=1`)

Quando a função precisa olhar **várias colunas da mesma linha**, use `axis=1`. A função recebe a linha inteira como uma Series, e você acessa cada coluna por nome: `linha['Salário']`.

In [36]:
def classificar_salario(linha):
    if linha['Salário'] >= 4000:
        return 'Alto'
    elif linha['Salário'] >= 3500:
        return 'Médio'
    else:
        return 'Inicial'
"""
Axis 0 (padrão): aplica por coluna a função recebe a coluna inteira

Axis 1: aplica por linha a função recebe a linha inteira
"""
df['Categoria'] = df.apply(classificar_salario, axis=1)
print(df)

     Nome  Idade  Salário  Salário com Aumento Categoria
0     Ana     25     3000               3300.0   Inicial
1   Bruno     30     4500               4950.0      Alto
2  Carlos     28     3800               4180.0     Médio


## Quando usar `axis`

Sempre que a lógica precisar olhar mais de uma coluna ao mesmo tempo. Por exemplo, se quisesse classificar combinando `idade` e `salário`:


```python
def classificar(linha):
    if linha['Idade'] < 30 and linha['Salário'] >= 4000:
        return 'Jovem bem pago'
    return 'Outros'

df['Status'] = df.apply(classificar, axis=1)
```


## 5. Função `lambda` (anônima)

`lambda` é um atalho para criar uma função de **uma única linha**, sem precisar do `def`. Boa quando a lógica é simples e não vale a pena dar nome.

`lambda x: x * 12` é equivalente a:
```python
def f(x):
    return x * 12
```

In [37]:
df['Idade em Meses'] = df['Idade'].apply(lambda x: x * 12)
print(df[['Nome', 'Idade', 'Idade em Meses']])

     Nome  Idade  Idade em Meses
0     Ana     25             300
1   Bruno     30             360
2  Carlos     28             336


## 6. `.map()` — substituir valores com um dicionário

`.map()` troca cada valor da coluna pelo correspondente no dicionário. Muito usado para criar colunas categóricas a partir de chaves (nome → cargo, sigla → estado, código → descrição).

In [38]:
mapa_cargo = {
    'Ana': 'Analista',
    'Bruno': 'Gerente',
    'Carlos': 'Coordenador'
}

df['Cargo'] = df['Nome'].map(mapa_cargo)
print(df[['Nome', 'Cargo']])

     Nome        Cargo
0     Ana     Analista
1   Bruno      Gerente
2  Carlos  Coordenador


## 7. Função customizada em `groupby`

Depois de agrupar com `.groupby()`, você pode passar sua própria função para `.agg()` — perfeito para métricas que o pandas não tem prontas, como a amplitude (max − min).

In [39]:
df['Departamento'] = ['Vendas', 'TI', 'Vendas']

def amplitude(serie):
    return serie.max() - serie.min()

resultado = df.groupby('Departamento')['Salário'].agg(amplitude)
print(resultado)

Departamento
TI          0
Vendas    800
Name: Salário, dtype: int64


## Resumo — qual usar e quando

| Padrão | Quando usar |
|---|---|
| `def f(df)` | Encapsular cálculos que recebem o DataFrame inteiro |
| `def f(df, x, y)` | Quando o cálculo precisa de parâmetros externos |
| `coluna.apply(f)` | Transformar valor a valor em **uma** coluna |
| `df.apply(f, axis=1)` | Função precisa de **várias colunas** da mesma linha |
| `lambda` | Lógica de uma linha, descartável |
| `.map(dicionario)` | Substituir valores por um mapeamento fixo |
| `groupby().agg(f)` | Métricas customizadas por grupo |

### Dica final
Sempre que se pegar escrevendo o mesmo trecho de código duas vezes, transforme em função. Isso facilita testar, dar nome ao que você está fazendo, e mudar a regra em um único lugar depois.